In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [21]:
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
class ScaledDotProductAttention(nn.Module):
    def __init__(self, d_k):
        super().__init__()
        self.d_k = d_k

    def forward(self, Q, K, V):
        scores = Q @ K.transpose(-2, -1)
        
        scores = scores / (self.d_k ** 0.5)

        scores = F.softmax(scores, dim=-1)

        output = scores @ V

        return output

In [4]:
class MultiHeadAttention(nn.Module):
    def __init__(self, h, d_model):
        super().__init__()
        self.h = h
        self.d_model = d_model
        self.d_k = self.d_model // self.h
        self.W_q = nn.Linear(self.d_model, self.d_model)
        self.W_k = nn.Linear(self.d_model, self.d_model)
        self.W_v = nn.Linear(self.d_model, self.d_model)
        self.W_o = nn.Linear(self.d_model, self.d_model)
        self.attention = ScaledDotProductAttention(self.d_k)

    def forward(self, X):
        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)

        batch_size = X.shape[0]
        Q = Q.view(batch_size, -1, self.h, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.h, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.h, self.d_k).transpose(1, 2)

        attention_output = self.attention(Q, K, V)

        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, -1, self.d_model)

        output = self.W_o(attention_output)

        return output


In [5]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        self.d_model = d_model
        self.d_ff = d_ff
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, X):
        X = self.linear1(X)
        X = F.relu(X)
        X = self.linear2(X)
        output = X

        return output
        

In [6]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len):
        super().__init__()

        pe = torch.zeros(max_seq_len, d_model)

        position = torch.arange(0, max_seq_len).unsqueeze(1)
        div_term = 10000 ** (torch.arange(0, d_model, 2) / d_model)


        pe[:, 0::2] = torch.sin(position / div_term)
        pe[:, 1::2] = torch.cos(position / div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, X):
        return X + self.pe[:, :X.size(1)]

In [7]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model, h, d_ff):
        super().__init__()
        self.attention = MultiHeadAttention(h, d_model)
        self.ff = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, X):
        attention_output = self.attention(X)
        X = self.norm1(attention_output + X)

        ff_output = self.ff(X)
        X = self.norm2(ff_output + X)

        return X
        

In [8]:
class Encoder(nn.Module):
    def __init__(self, h, d_model, vocab_size, d_ff, N, max_seq_len):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_seq_len)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, h, d_ff) for _ in range(N)])

    def forward(self, X):
        X = self.embedding(X)
        X = self.pos_encoding(X)

        for block in self.blocks:
            X = block(X)

        return X

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = Encoder(
    h=4,
    d_model=128,
    vocab_size=10000,
    d_ff=512,
    N=2,
    max_seq_len=128
).to(device)

X = torch.randint(0, 10000, (32, 128)).to(device)

output = model(X)

print("Device:", device)
print("Input shape:", X.shape)
print("Output shape:", output.shape)

Device: cuda
Input shape: torch.Size([32, 128])
Output shape: torch.Size([32, 128, 128])


In [10]:
class SentimentClassifier(nn.Module):
    def __init__(self, Encoder, d_model):
        super().__init__()
        self.Encoder = Encoder
        self.d_model = d_model
        self.fc = nn.Linear(d_model, 2)

    def forward(self, X):
        X = self.Encoder(X)
        X = X.mean(dim=1)
        X = self.fc(X)
        return X

In [12]:
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")

def tokenizer(text):
    return text.lower().split()


def build_vocab(dataset):
    vocab = {'<PAD>': 0, '<UNK>': 1}

    unique_words = set()
    for example in dataset['train']:
        words = tokenizer(example['text'])
        unique_words.update(words)

    for i, word in enumerate(unique_words, start=2):
        vocab[word] = i 

    return vocab
        

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [13]:
vocab = build_vocab(dataset)
print(len(vocab))

251639


In [19]:
from torch.utils.data import Dataset, DataLoader

MAX_LEN    = 256
D_MODEL    = 128
H          = 4
D_FF       = 256
N          = 2
BATCH_SIZE = 32
EPOCHS     = 3
LR         = 1e-3
VOCAB_SIZE = len(vocab)
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def text_to_ids(text, max_len=MAX_LEN):
    tokens = tokenizer(text)
    ids = [vocab.get(word, vocab['<UNK>']) for word in tokens]
    ids = ids[:max_len]                               # truncate
    ids = ids + [vocab['<PAD>']] * (max_len - len(ids))  # pad
    return ids

class IMDBDataset(Dataset):
    def __init__(self, split):
        self.data = dataset[split]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ids   = text_to_ids(self.data[idx]['text'])
        label = self.data[idx]['label']
        return torch.tensor(ids, dtype=torch.long), torch.tensor(label, dtype=torch.long)

train_loader = DataLoader(IMDBDataset('train'), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(IMDBDataset('test'),  batch_size=BATCH_SIZE, shuffle=False)

encoder = Encoder(
    h=H, d_model=D_MODEL, vocab_size=VOCAB_SIZE,
    d_ff=D_FF, N=N, max_seq_len=MAX_LEN
)
model = SentimentClassifier(encoder, D_MODEL).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

def train_epoch(model, loader):
    model.train()
    total_loss, correct, total = 0, 0, 0
    for i, (X, y) in enumerate(loader):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        out  = model(X)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        correct    += (out.argmax(1) == y).sum().item()
        total      += y.size(0)
        if i % 100 == 0:
            print(f"  batch {i}/{len(loader)}  loss: {loss.item():.4f}")
    return total_loss / len(loader), correct / total

def evaluate(model, loader):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            out  = model(X)
            loss = criterion(out, y)
            total_loss += loss.item()
            correct    += (out.argmax(1) == y).sum().item()
            total      += y.size(0)
    return total_loss / len(loader), correct / total

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    tr_loss, tr_acc = train_epoch(model, train_loader)
    te_loss, te_acc = evaluate(model, test_loader)
    print(f"Train  loss: {tr_loss:.4f}  acc: {tr_acc:.4f}")
    print(f"Test   loss: {te_loss:.4f}  acc: {te_acc:.4f}")
    print("-" * 40)


Epoch 1/3
  batch 0/782  loss: 0.7574
  batch 100/782  loss: 0.6009
  batch 200/782  loss: 0.6190
  batch 300/782  loss: 0.6322
  batch 400/782  loss: 0.5802
  batch 500/782  loss: 0.4660
  batch 600/782  loss: 0.3470
  batch 700/782  loss: 0.4662
Train  loss: 0.5337  acc: 0.7194
Test   loss: 0.4120  acc: 0.8098
----------------------------------------

Epoch 2/3
  batch 0/782  loss: 0.3250
  batch 100/782  loss: 0.4195
  batch 200/782  loss: 0.2458
  batch 300/782  loss: 0.2560
  batch 400/782  loss: 0.3403
  batch 500/782  loss: 0.4132
  batch 600/782  loss: 0.3445
  batch 700/782  loss: 0.4071
Train  loss: 0.3240  acc: 0.8622
Test   loss: 0.4425  acc: 0.8104
----------------------------------------

Epoch 3/3
  batch 0/782  loss: 0.0585
  batch 100/782  loss: 0.1145
  batch 200/782  loss: 0.1511
  batch 300/782  loss: 0.2129
  batch 400/782  loss: 0.2896
  batch 500/782  loss: 0.1306
  batch 600/782  loss: 0.2030
  batch 700/782  loss: 0.1543
Train  loss: 0.1786  acc: 0.9322
Test  

In [15]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, hidden_size, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.lstm = nn.LSTM(d_model, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, X):
        X = self.embedding(X)
        _, (hidden, _) = self.lstm(X)
        X = self.fc(hidden[-1])
        return X


class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, d_model, hidden_size, num_layers):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.gru = nn.GRU(d_model, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, X):
        X = self.embedding(X)
        _, hidden = self.gru(X)
        X = self.fc(hidden[-1])
        return X


def train_model(model, train_loader, test_loader, epochs=3, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()
    
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0, 0, 0
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct += (out.argmax(1) == y).sum().item()
            total += y.size(0)
        
        model.eval()
        te_loss, te_correct, te_total = 0, 0, 0
        with torch.no_grad():
            for X, y in test_loader:
                X, y = X.to(device), y.to(device)
                out = model(X)
                te_loss += criterion(out, y).item()
                te_correct += (out.argmax(1) == y).sum().item()
                te_total += y.size(0)
        
        print(f"Epoch {epoch+1} | Train acc: {correct/total:.4f} | Test acc: {te_correct/te_total:.4f}")


print("Training LSTM...")
lstm_model = LSTMClassifier(VOCAB_SIZE, D_MODEL, hidden_size=128, num_layers=2).to(device)
train_model(lstm_model, train_loader, test_loader)

print("\nTraining GRU...")
gru_model = GRUClassifier(VOCAB_SIZE, D_MODEL, hidden_size=128, num_layers=2).to(device)
train_model(gru_model, train_loader, test_loader)

Training LSTM...
Epoch 1 | Train acc: 0.5028 | Test acc: 0.4979
Epoch 2 | Train acc: 0.5082 | Test acc: 0.5009
Epoch 3 | Train acc: 0.5355 | Test acc: 0.5204

Training GRU...
Epoch 1 | Train acc: 0.5635 | Test acc: 0.6655
Epoch 2 | Train acc: 0.8372 | Test acc: 0.8397
Epoch 3 | Train acc: 0.9389 | Test acc: 0.8370


In [16]:
results = {
    "Transformer": 0.8247,
    "GRU": 0.8397,
    "LSTM": 0.5204
}

winner = max(results, key=lambda x: results[x])
print(f"{winner} wins with {results[winner]*100:.2f}% test accuracy!")

GRU wins with 83.97% test accuracy!
